In [ ]:
import pandas as pd
from pathlib import Path

# Loading raw playoff data
Mypath = (Path.home()/ "Desktop"/ "NBA_Project"/ "data"/ "raw" / "playoff_series_history.xls")
tables = pd.read_html(Mypath)
playoff_series = tables[0].copy()

# Selecting useful columns
playoff_series_clean = playoff_series.iloc[:, [0, 1, 2, 3, 5, 6, 8, 9]].copy()

# Renaming columns
playoff_series_clean.columns = [
    "year",
    "league",
    "series",
    "date_range",
    "winner_team",
    "winner_wins",
    "loser_team",
    "loser_wins"
]

# Filtering NBA playoff series from 1997 to 2026
playoff_series_clean = playoff_series_clean[
    (playoff_series_clean["league"] == "NBA") &
    (playoff_series_clean["year"] >= 1997) &
    (playoff_series_clean["year"] <= 2026)
].copy()

print("Playoff series shape:", playoff_series_clean.shape)

playoff_series_clean.head()

Playoff series shape: (450, 8)


,year,league,series,date_range,winner_team,winner_wins,loser_team,loser_wins
0,2026,NBA,Eastern Conf First Round,"Apr 19 - May 3, 2026",Detroit Pistons (1),4,Orlando Magic (8),3
1,2026,NBA,Eastern Conf First Round,"Apr 18 - May 3, 2026",Cleveland Cavaliers (4),4,Toronto Raptors (5),3
2,2026,NBA,Eastern Conf First Round,"Apr 18 - Apr 30, 2026",New York Knicks (3),4,Atlanta Hawks (6),2
3,2026,NBA,Eastern Conf First Round,"Apr 19 - May 2, 2026",Philadelphia 76ers (7),4,Boston Celtics (2),3
4,2026,NBA,Western Conf First Round,"Apr 19 - Apr 27, 2026",Oklahoma City Thunder (1),4,Phoenix Suns (8),0


In [ ]:
# Creating playoff team results from playoff series data

playoff_series_work = playoff_series_clean.copy()

# Extracting playoff seeds
playoff_series_work["winner_seed"] = playoff_series_work["winner_team"].astype(str).str.extract(r"\((\d+)\)").astype("Int64")
playoff_series_work["loser_seed"] = playoff_series_work["loser_team"].astype(str).str.extract(r"\((\d+)\)").astype("Int64")

# Removing playoff seeds from team names
playoff_series_work["winner_team"] = playoff_series_work["winner_team"].astype(str).str.replace(r"\s*\(\d+\)", "", regex=True).str.strip()
playoff_series_work["loser_team"] = playoff_series_work["loser_team"].astype(str).str.replace(r"\s*\(\d+\)", "", regex=True).str.strip()

# Adding conference information
def get_conference_from_series(series_name):
    if "Eastern" in str(series_name):
        return "East"
    elif "Western" in str(series_name):
        return "West"
    return pd.NA

playoff_series_work["conference"] = playoff_series_work["series"].apply(get_conference_from_series)

# Creating conference lookup table
winner_conference = playoff_series_work[["year","winner_team","conference"]].rename(columns={"winner_team":"team_full_name"})
loser_conference = playoff_series_work[["year","loser_team","conference"]].rename(columns={"loser_team":"team_full_name"})

team_conference_lookup = pd.concat(
    [winner_conference, loser_conference],
    ignore_index=True
).dropna(subset=["conference"]).drop_duplicates(["year","team_full_name"])

# Creating loser playoff results
def get_loser_playoff_stage(series_name):
    if series_name == "Finals":
        return "Finals Runner-up"
    elif "Conf Finals" in str(series_name):
        return "Conference Finals"
    elif "Semifinals" in str(series_name):
        return "Conference Semifinals"
    elif "First Round" in str(series_name):
        return "First Round"
    return "Unknown"

loser_results = playoff_series_work[[
    "year",
    "loser_team",
    "loser_seed",
    "series"
]].rename(columns={
    "loser_team":"team_full_name",
    "loser_seed":"playoff_seed"
})

loser_results["playoff_stage"] = loser_results["series"].apply(get_loser_playoff_stage)

# Creating champion results
champion_results = playoff_series_work[
    playoff_series_work["series"] == "Finals"
][[
    "year",
    "winner_team",
    "winner_seed"
]].rename(columns={
    "winner_team":"team_full_name",
    "winner_seed":"playoff_seed"
})

champion_results["playoff_stage"] = "Champion"

# Combining playoff results
playoff_team_results = pd.concat(
    [
        loser_results[["year","team_full_name","playoff_seed","playoff_stage"]],
        champion_results[["year","team_full_name","playoff_seed","playoff_stage"]]
    ],
    ignore_index=True
)

# Adding conference information
playoff_team_results = playoff_team_results.merge(
    team_conference_lookup,
    on=["year","team_full_name"],
    how="left"
)

# Adding season and playoff indicators
playoff_team_results["season"] = (
    (playoff_team_results["year"] - 1).astype(str)
    + "-"
    + playoff_team_results["year"].astype(str).str[-2:]
)

playoff_team_results["actual_playoff_team"] = 1
playoff_team_results["top_8_conference_team"] = 1

playoff_team_results["won_nba_title"] = (
    playoff_team_results["playoff_stage"] == "Champion"
).astype(int)

playoff_team_results["reached_finals"] = (
    playoff_team_results["playoff_stage"].isin(
        ["Champion","Finals Runner-up"]
    )
).astype(int)

playoff_team_results["reached_conference_finals"] = (
    playoff_team_results["playoff_stage"].isin(
        ["Champion","Finals Runner-up","Conference Finals"]
    )
).astype(int)

playoff_team_results["reached_conference_semifinals"] = (
    playoff_team_results["playoff_stage"].isin(
        ["Champion","Finals Runner-up","Conference Finals","Conference Semifinals"]
    )
).astype(int)

# Adding playoff rank
playoff_stage_rank_map = {
    "Champion":1,
    "Finals Runner-up":2,
    "Conference Finals":3,
    "Conference Semifinals":4,
    "First Round":5
}

playoff_team_results["playoff_finish_rank"] = (
    playoff_team_results["playoff_stage"].map(playoff_stage_rank_map)
).astype("Int64")

# Reordering columns
playoff_team_results = playoff_team_results[[
    "year",
    "season",
    "team_full_name",
    "conference",
    "playoff_seed",
    "actual_playoff_team",
    "top_8_conference_team",
    "playoff_stage",
    "playoff_finish_rank",
    "reached_conference_semifinals",
    "reached_conference_finals",
    "reached_finals",
    "won_nba_title"
]]

playoff_team_results = playoff_team_results.sort_values(
    [
        "year",
        "conference",
        "playoff_seed",
        "playoff_finish_rank",
        "team_full_name"
    ]
).copy()

display(playoff_team_results.head(50))

# Saving playoff team results
toPath = Path.home() / "Desktop" / "NBA_Project" / "data" / "processed"
toPath.mkdir(parents=True, exist_ok=True)

output_file = toPath / "playoff_team_results_clean.csv"
playoff_team_results.to_csv(output_file, index=False)

print("Saved playoff team results to:", output_file)

,year,season,team_full_name,conference,playoff_seed,actual_playoff_team,top_8_conference_team,playoff_stage,playoff_finish_rank,reached_conference_semifinals,reached_conference_finals,reached_finals,won_nba_title
479,1997,1996-97,Chicago Bulls,East,1,1,1,Champion,1,1,1,1,1
447,1997,1996-97,Miami Heat,East,2,1,1,Conference Finals,3,1,1,0,0
444,1997,1996-97,New York Knicks,East,3,1,1,Conference Semifinals,4,1,0,0,0
443,1997,1996-97,Atlanta Hawks,East,4,1,1,Conference Semifinals,4,1,0,0,0
435,1997,1996-97,Detroit Pistons,East,5,1,1,First Round,5,0,0,0,0
438,1997,1996-97,Charlotte Hornets,East,6,1,1,First Round,5,0,0,0,0
437,1997,1996-97,Orlando Magic,East,7,1,1,First Round,5,0,0,0,0
436,1997,1996-97,Washington Bullets,East,8,1,1,First Round,5,0,0,0,0
449,1997,1996-97,Utah Jazz,West,1,1,1,Finals Runner-up,2,1,1,1,0
445,1997,1996-97,Seattle SuperSonics,West,2,1,1,Conference Semifinals,4,1,0,0,0


Saved playoff team results to: C:\Users\ahmed\Desktop\NBA_Project\data\processed\playoff_team_results_clean.csv
